# OptiCrop - Crop Recommendation Analysis & Modeling
This notebook provides the complete exploratory data analysis (EDA) and machine learning modeling pipeline for the **OptiCrop** smart agricultural production optimization engine. 

### Objectives
- **Recommend** the most suitable crop based on soil nutrients and climatic factors.
- **Perform EDA** to understand relationships between features (N, P, K, temperature, humidity, pH, rainfall).
- **Train and Evaluate** multiple supervised models (Logistic Regression, KNN, Decision Tree, Random Forest) and unsupervised clustering (K-Means).
- **Select and Serialize** the best performing classification model.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Set style for plotting
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load and Inspect Dataset

In [ ]:
# Load dataset from the local path or remote URL
dataset_path = "../dataset/crop_recommendation.csv"
if not os.path.exists(dataset_path):
    dataset_path = "https://raw.githubusercontent.com/PAIshanMadusha/crop-recommendation-model/main/dataset/Crop_recommendation.csv"
    
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Preprocessing & Exploratory Data Analysis (EDA)
Let's inspect values, missing parameters, duplicate rows, and correlation coefficients.

In [ ]:
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Duplicated Rows Check ---")
print(f"Duplicate Rows: {df.duplicated().sum()}")

print("\n--- Unique Target Classes ---")
print(df['label'].value_counts())

### Heatmap of Correlation between features

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.drop(columns=['label']).corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

### Distributions of each soil & climate feature

In [ ]:
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, feat in enumerate(features):
    sns.histplot(df[feat], ax=axes[i], kde=True, color='forestgreen')
    axes[i].set_title(f"Distribution of {feat}")
    
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

## 3. Train-Test Split and Preprocessing Pipeline

In [ ]:
X = df.drop(columns=['label'])
y = df['label']

# Encode the target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. Supervised Model Training & Evaluation

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cv_score = cross_val_score(model, X_train, y_train, cv=5).mean()
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1,
        "Cross Validation Accuracy": cv_score
    })
    
results_df = pd.DataFrame(results)
results_df

## 5. Unsupervised Clustering with K-Means

In [ ]:
# Train K-Means with 22 clusters (since we have 22 crop classes)
kmeans = KMeans(n_clusters=22, random_state=42, n_init=10)
kmeans.fit(X_scaled)
df['cluster'] = kmeans.labels_

print("Cluster counts:")
print(df['cluster'].value_counts().head())

## 6. Selection & Serialization of Best Model

In [ ]:
best_row = results_df.loc[results_df['Accuracy'].idxmax()]
print(f"Best Model: {best_row['Model']} with Accuracy {best_row['Accuracy']:.4f}")